# 3.1 Data Selection — Titanic Survival Prediction

> **Author:** Thibauld  
> **Date:** 2026-03-31  
> **CRISP-DM Phase:** 3. Data Preparation  
> **Purpose:** Decide which datasets, fields, and records to carry forward into the modeling dataset, with documented rationale for every inclusion/exclusion decision.
>
> **Source documents:** [1.3 Data Mining Goals](../docs/crisp-dm/1-business-understanding/1.3-data-mining-goals.md), [2.1 Data Collection](../docs/crisp-dm/2-data-understanding/2.1-data-collection.md), [2.2 Data Description](../docs/crisp-dm/2-data-understanding/2.2-data-description.md), [2.3 Data Exploration](../docs/crisp-dm/2-data-understanding/2.3-data-exploration.md), [2.4 Data Quality](../docs/crisp-dm/2-data-understanding/2.4-data-quality.md)

In [1]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path(__file__).resolve().parent.parent if "__file__" in dir() else Path.cwd()
if (PROJECT_ROOT / "notebooks").is_dir():
    pass  # cwd is project root
elif (PROJECT_ROOT.parent / "notebooks").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent  # cwd is a subdirectory

DATA_DIR = PROJECT_ROOT / "data" / "raw" / "titanic"

# Load all datasets from Phase 2
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
baseline = pd.read_csv(DATA_DIR / "gender_submission.csv")

print(f"train.csv:              {train.shape[0]:>4} rows × {train.shape[1]:>2} cols")
print(f"test.csv:               {test.shape[0]:>4} rows × {test.shape[1]:>2} cols")
print(f"gender_submission.csv:  {baseline.shape[0]:>4} rows × {baseline.shape[1]:>2} cols")

train.csv:               891 rows × 12 cols
test.csv:                418 rows × 11 cols
gender_submission.csv:   418 rows ×  2 cols


## 1. Dataset-Level Selection

Three datasets are available from Phase 2 (task 2.1). Each is assessed against the data mining goals from task 1.3.

| Dataset | Decision | Role | Data Mining Goals | Rationale |
|---------|----------|------|-------------------|-----------|
| **train.csv** | ✅ Include | Training + CV | DM1, DM2, DM3, DM4 | Contains target variable (`Survived`) and all feature columns. Required for model training, feature importance (DM1), classification (DM2), model comparison (DM3), and ablation (DM4). |
| **test.csv** | ✅ Include | Prediction | DM2 | Contains features for 418 passengers whose survival must be predicted for Kaggle submission. |
| **gender_submission.csv** | ✅ Include (reference) | Baseline | DM2 | Submission format template and gender-only baseline benchmark (~76.5% accuracy). Not used as training data. |

**No datasets excluded.** All three serve distinct, necessary roles.

In [2]:
# Verify datasets are loadable and structurally sound
datasets = {"train.csv": train, "test.csv": test, "gender_submission.csv": baseline}

summary = pd.DataFrame(
    [
        {
            "Dataset": name,
            "Rows": df.shape[0],
            "Columns": df.shape[1],
            "Memory (KB)": round(df.memory_usage(deep=True).sum() / 1024, 1),
            "Duplicates": df.duplicated().sum(),
            "Has Target": "Survived" in df.columns,
        }
        for name, df in datasets.items()
    ]
)
summary

,Dataset,Rows,Columns,Memory (KB),Duplicates,Has Target
0,train.csv,891,12,285.6,0,True
1,test.csv,418,11,131.0,0,False
2,gender_submission.csv,418,2,6.7,0,True


## 2. Field-Level Selection

Every field in `train.csv` (and its counterpart in `test.csv`) is assessed for its role in modeling. The roles are:
- **Target** — the variable to predict
- **ID** — identifier, not used as a feature but required for submission
- **Feature** — used directly as a model input
- **Engineering source** — raw field not used directly, but used to derive engineered features
- **Excluded** — dropped from the modeling dataset (with rationale)

### 2.1 Field Role Assignment

In [3]:
# Field selection registry — maps every field to its role and rationale
field_selection = pd.DataFrame(
    [
        {
            "Field": "PassengerId",
            "Type": "int64",
            "Decision": "Include",
            "Role": "ID",
            "Rationale": "Required for Kaggle submission mapping. Not used as a model feature.",
            "DM Goal": "DM2",
        },
        {
            "Field": "Survived",
            "Type": "int64",
            "Decision": "Include",
            "Role": "Target",
            "Rationale": "Binary classification target (0=deceased, 1=survived). Present in train only.",
            "DM Goal": "DM1-DM4",
        },
        {
            "Field": "Pclass",
            "Type": "int64",
            "Decision": "Include",
            "Role": "Feature",
            "Rationale": "Strong predictor (Cramer's V=0.340 with Survived). Ordinal: 1st>2nd>3rd. Critical Sex×Pclass interaction.",
            "DM Goal": "DM1, DM2",
        },
        {
            "Field": "Name",
            "Type": "object",
            "Decision": "Include",
            "Role": "Engineering source",
            "Rationale": "Title extraction (Mr/Mrs/Miss/Master) yields 15.7%–79.2% survival range. Raw Name not used directly.",
            "DM Goal": "DM2, DM4",
        },
        {
            "Field": "Sex",
            "Type": "object",
            "Decision": "Include",
            "Role": "Feature",
            "Rationale": "Strongest single predictor (Cramer's V=0.541). Female 74.2% vs Male 18.9% survival.",
            "DM Goal": "DM1, DM2",
        },
        {
            "Field": "Age",
            "Type": "float64",
            "Decision": "Include",
            "Role": "Feature",
            "Rationale": "Non-linear survival effect (infants 70.5%, elderly 22.7%). 20% missing (MAR) — impute in 3.2.",
            "DM Goal": "DM1, DM2, DM4",
        },
        {
            "Field": "SibSp",
            "Type": "int64",
            "Decision": "Include",
            "Role": "Engineering source",
            "Rationale": "Component of FamilySize (SibSp+Parch+1). Zero-inflated but contributes to family structure signal.",
            "DM Goal": "DM2, DM4",
        },
        {
            "Field": "Parch",
            "Type": "int64",
            "Decision": "Include",
            "Role": "Engineering source",
            "Rationale": "Component of FamilySize. Same as SibSp — combined they reveal inverted-U survival pattern.",
            "DM Goal": "DM2, DM4",
        },
        {
            "Field": "Ticket",
            "Type": "object",
            "Decision": "Include",
            "Role": "Engineering source",
            "Rationale": "Shared tickets identify travel groups (ticket group size mirrors family size pattern). Raw Ticket not used directly.",
            "DM Goal": "DM2, DM4",
        },
        {
            "Field": "Fare",
            "Type": "float64",
            "Decision": "Include",
            "Role": "Feature",
            "Rationale": "Moderate predictor (rank-biserial r=0.384). Extreme right skew — log-transform in 3.3. 1 missing in test.",
            "DM Goal": "DM1, DM2",
        },
        {
            "Field": "Cabin",
            "Type": "object",
            "Decision": "Include",
            "Role": "Engineering source",
            "Rationale": "77% missing (MNAR) — too sparse for direct use. HasCabin (66.7% vs 30.0% survival) and Deck extraction valuable.",
            "DM Goal": "DM2, DM4",
        },
        {
            "Field": "Embarked",
            "Type": "object",
            "Decision": "Include",
            "Role": "Feature",
            "Rationale": "Weak-moderate predictor (Cramer's V=0.172). Cherbourg higher survival likely confounded by class. 2 missing in train.",
            "DM Goal": "DM2",
        },
    ]
)

# Display with styling
field_selection.style.set_properties(**{"text-align": "left"}).hide(axis="index")

Field,Type,Decision,Role,Rationale,DM Goal
PassengerId,int64,Include,ID,Required for Kaggle submission mapping. Not used as a model feature.,DM2
Survived,int64,Include,Target,"Binary classification target (0=deceased, 1=survived). Present in train only.",DM1-DM4
Pclass,int64,Include,Feature,Strong predictor (Cramer's V=0.340 with Survived). Ordinal: 1st>2nd>3rd. Critical Sex×Pclass interaction.,"DM1, DM2"
Name,object,Include,Engineering source,Title extraction (Mr/Mrs/Miss/Master) yields 15.7%–79.2% survival range. Raw Name not used directly.,"DM2, DM4"
Sex,object,Include,Feature,Strongest single predictor (Cramer's V=0.541). Female 74.2% vs Male 18.9% survival.,"DM1, DM2"
Age,float64,Include,Feature,"Non-linear survival effect (infants 70.5%, elderly 22.7%). 20% missing (MAR) — impute in 3.2.","DM1, DM2, DM4"
SibSp,int64,Include,Engineering source,Component of FamilySize (SibSp+Parch+1). Zero-inflated but contributes to family structure signal.,"DM2, DM4"
Parch,int64,Include,Engineering source,Component of FamilySize. Same as SibSp — combined they reveal inverted-U survival pattern.,"DM2, DM4"
Ticket,object,Include,Engineering source,Shared tickets identify travel groups (ticket group size mirrors family size pattern). Raw Ticket not used directly.,"DM2, DM4"
Fare,float64,Include,Feature,Moderate predictor (rank-biserial r=0.384). Extreme right skew — log-transform in 3.3. 1 missing in test.,"DM1, DM2"


In [4]:
# Field selection summary
roles = field_selection["Role"].value_counts()
total_fields = len(field_selection)

print("Field Selection Summary (train.csv)")
print("=" * 40)
for role, count in roles.items():
    print(f"  {role:<25s} {count:>2} fields")
print(f"  {'TOTAL':<25s} {total_fields:>2} fields")
print(f"\nExcluded fields: 0")
print(f"Retention rate: {total_fields}/{total_fields} = 100%")

Field Selection Summary (train.csv)
  Feature                    5 fields
  Engineering source         5 fields
  ID                         1 fields
  Target                     1 fields
  TOTAL                     12 fields

Excluded fields: 0
Retention rate: 12/12 = 100%


### 2.2 Missingness Profile of Selected Fields

Quality issues from 2.4 that affect field usability:

In [5]:
# Missingness profile for selected fields across train and test
missing_train = train.isnull().sum()
missing_test = test.isnull().sum()

missing_profile = pd.DataFrame(
    {
        "Field": train.columns,
        "Train Missing": missing_train.values,
        "Train Missing %": (missing_train.values / len(train) * 100).round(1),
        "Test Missing": [missing_test.get(col, "N/A") for col in train.columns],
        "Test Missing %": [
            round(missing_test.get(col, 0) / len(test) * 100, 1)
            if col in test.columns
            else "N/A"
            for col in train.columns
        ],
    }
)

# Only show fields with any missingness
has_missing = missing_profile[
    (missing_profile["Train Missing"] > 0)
    | (missing_profile["Test Missing"].apply(lambda x: x != "N/A" and x > 0))
]
has_missing

,Field,Train Missing,Train Missing %,Test Missing,Test Missing %
5,Age,177,19.9,86,20.6
9,Fare,0,0.0,1,0.2
10,Cabin,687,77.1,327,78.2
11,Embarked,2,0.2,0,0.0


### 2.3 Data Leakage Assessment

All features represent pre-event passenger attributes (cabin assignment, ticket purchase, embarkation port). None encode post-event information or the target variable itself.

In [6]:
# Data leakage checks
print("Data Leakage Assessment")
print("=" * 60)

# 1. Train/test overlap check
train_ids = set(train["PassengerId"])
test_ids = set(test["PassengerId"])
overlap = train_ids & test_ids
print(f"\n1. Train/test PassengerId overlap: {len(overlap)} (expected: 0)")

# 2. Check that test has no target
print(f"2. 'Survived' in test columns: {'Survived' in test.columns} (expected: False)")

# 3. Check baseline alignment with test
baseline_ids = set(baseline["PassengerId"])
print(f"3. Baseline PassengerId matches test: {baseline_ids == test_ids} (expected: True)")

# 4. Feature temporality assessment
print("\n4. Feature Temporality Assessment:")
leakage_check = [
    ("Pclass", "Pre-event (ticket class at purchase)", "Safe"),
    ("Name", "Pre-event (passenger identity)", "Safe"),
    ("Sex", "Pre-event (demographic)", "Safe"),
    ("Age", "Pre-event (demographic)", "Safe"),
    ("SibSp", "Pre-event (family structure at boarding)", "Safe"),
    ("Parch", "Pre-event (family structure at boarding)", "Safe"),
    ("Ticket", "Pre-event (purchased before voyage)", "Safe"),
    ("Fare", "Pre-event (paid at booking)", "Safe"),
    ("Cabin", "Pre-event (assigned at boarding)", "Safe"),
    ("Embarked", "Pre-event (port of departure)", "Safe"),
]
for field, timing, status in leakage_check:
    print(f"   {field:<12s} {timing:<45s} [{status}]")

# 5. Preprocessing leakage guard
print("\n5. Preprocessing Leakage Guard:")
print("   CRITICAL: All imputers, encoders, and scalers must be fit on")
print("   train data ONLY, then applied (transform) to test data.")
print("   Never fit on combined train+test.")

Data Leakage Assessment

1. Train/test PassengerId overlap: 0 (expected: 0)
2. 'Survived' in test columns: False (expected: False)
3. Baseline PassengerId matches test: True (expected: True)

4. Feature Temporality Assessment:
   Pclass       Pre-event (ticket class at purchase)          [Safe]
   Name         Pre-event (passenger identity)                [Safe]
   Sex          Pre-event (demographic)                       [Safe]
   Age          Pre-event (demographic)                       [Safe]
   SibSp        Pre-event (family structure at boarding)      [Safe]
   Parch        Pre-event (family structure at boarding)      [Safe]
   Ticket       Pre-event (purchased before voyage)           [Safe]
   Fare         Pre-event (paid at booking)                   [Safe]
   Cabin        Pre-event (assigned at boarding)              [Safe]
   Embarked     Pre-event (port of departure)                 [Safe]

5. Preprocessing Leakage Guard:
   CRITICAL: All imputers, encoders, and scalers m

## 3. Record-Level Selection

With only 891 training rows and 418 test rows, every observation is valuable. This section verifies that no records need to be excluded and documents the rationale.

In [7]:
# Record-level selection analysis
print("Record-Level Selection Analysis")
print("=" * 60)

# 1. Row completeness distribution
fields_per_row_train = train.notnull().sum(axis=1)
fields_per_row_test = test.notnull().sum(axis=1)
missing_per_row_train = train.isnull().sum(axis=1)

print("\n1. Row completeness (train.csv):")
completeness_dist = missing_per_row_train.value_counts().sort_index()
for n_missing, count in completeness_dist.items():
    pct = count / len(train) * 100
    print(f"   {n_missing} fields missing: {count:>3} rows ({pct:.1f}%)")

print(f"\n   Rows with >50% fields missing: {(missing_per_row_train > 6).sum()}")
print(f"   Minimum fields present per row: {fields_per_row_train.min()} / {train.shape[1]}")

# 2. Edge case records
print("\n2. Edge Case Records (potential exclusion candidates):")

# Zero-fare passengers
zero_fare_train = train[train["Fare"] == 0]
print(f"\n   a) Zero-fare passengers (train): {len(zero_fare_train)}")
print(f"      All male: {(zero_fare_train['Sex'] == 'male').all()}")
print(f"      Survival rate: {zero_fare_train['Survived'].mean():.1%}")
print(f"      Decision: KEEP — likely valid edge cases (crew, journalists, special)")

# Large families (FamilySize >= 8)
train_fs = train["SibSp"] + train["Parch"] + 1
large_families = train[train_fs >= 8]
print(f"\n   b) Large families (FamilySize >= 8): {len(large_families)}")
print(f"      Survival rate: {large_families['Survived'].mean():.1%}")
print(f"      Decision: KEEP — genuine signal (large families struggled to evacuate)")

# Passengers with max missing (Cabin + Age)
max_missing = train[missing_per_row_train == missing_per_row_train.max()]
print(f"\n   c) Rows with max missing ({missing_per_row_train.max()} fields): {len(max_missing)}")
print(f"      Missing fields: Cabin + Age (always)")
print(f"      Decision: KEEP — imputation in 3.2 will handle these")

print("\n3. Record Selection Decision:")
print("   NO RECORDS EXCLUDED from either train or test.")
print(f"   Train: {len(train)}/{len(train)} rows retained (100%)")
print(f"   Test:  {len(test)}/{len(test)} rows retained (100%)")

Record-Level Selection Analysis

1. Row completeness (train.csv):
   0 fields missing: 183 rows (20.5%)
   1 fields missing: 550 rows (61.7%)
   2 fields missing: 158 rows (17.7%)

   Rows with >50% fields missing: 0
   Minimum fields present per row: 10 / 12

2. Edge Case Records (potential exclusion candidates):

   a) Zero-fare passengers (train): 15
      All male: True
      Survival rate: 6.7%
      Decision: KEEP — likely valid edge cases (crew, journalists, special)

   b) Large families (FamilySize >= 8): 13
      Survival rate: 0.0%
      Decision: KEEP — genuine signal (large families struggled to evacuate)

   c) Rows with max missing (2 fields): 158
      Missing fields: Cabin + Age (always)
      Decision: KEEP — imputation in 3.2 will handle these

3. Record Selection Decision:
   NO RECORDS EXCLUDED from either train or test.
   Train: 891/891 rows retained (100%)
   Test:  418/418 rows retained (100%)


## 4. Coverage Analysis

Final dataset dimensions after all selection decisions.

In [8]:
# Coverage analysis
print("Coverage Analysis")
print("=" * 60)

print("\nDataset Coverage:")
print(f"  Datasets selected:    3 / 3 (100%)")
print(f"  Train records:        {len(train)} / {len(train)} (100% retained)")
print(f"  Test records:         {len(test)} / {len(test)} (100% retained)")
print(f"  Total records:        {len(train) + len(test)} / {len(train) + len(test)} (100%)")

print("\nField Coverage:")
direct_features = field_selection[field_selection["Role"] == "Feature"]
eng_sources = field_selection[field_selection["Role"] == "Engineering source"]
print(f"  Total fields (train): {len(field_selection)} / {train.shape[1]} (100% retained)")
print(f"  Direct features:      {len(direct_features)}")
print(f"  Engineering sources:  {len(eng_sources)}")
print(f"  Target:               1 (Survived)")
print(f"  ID:                   1 (PassengerId)")
print(f"  Excluded:             0")

# Planned engineered features (to be created in 3.3)
print("\nPlanned Engineered Features (to create in task 3.3):")
planned_features = [
    ("Title", "Name", "Extract Mr/Mrs/Miss/Master from Name field"),
    ("FamilySize", "SibSp + Parch", "SibSp + Parch + 1"),
    ("FamilySizeBin", "FamilySize", "Small(1) / Medium(2-4) / Large(5+)"),
    ("IsAlone", "FamilySize", "Binary: FamilySize == 1"),
    ("HasCabin", "Cabin", "Binary: Cabin is not null"),
    ("Deck", "Cabin", "First letter of Cabin (A-G)"),
    ("AgeMissing", "Age", "Binary: Age is null (before imputation)"),
    ("log(Fare+1)", "Fare", "Log-transform to reduce skew"),
]
for name, source, desc in planned_features:
    print(f"  {name:<16s} ← {source:<14s}  {desc}")

print(f"\n  Total features after engineering: {len(direct_features) + len(planned_features)} "
      f"({len(direct_features)} direct + {len(planned_features)} engineered)")

Coverage Analysis

Dataset Coverage:
  Datasets selected:    3 / 3 (100%)
  Train records:        891 / 891 (100% retained)
  Test records:         418 / 418 (100% retained)
  Total records:        1309 / 1309 (100%)

Field Coverage:
  Total fields (train): 12 / 12 (100% retained)
  Direct features:      5
  Engineering sources:  5
  Target:               1 (Survived)
  ID:                   1 (PassengerId)
  Excluded:             0

Planned Engineered Features (to create in task 3.3):
  Title            ← Name            Extract Mr/Mrs/Miss/Master from Name field
  FamilySize       ← SibSp + Parch   SibSp + Parch + 1
  FamilySizeBin    ← FamilySize      Small(1) / Medium(2-4) / Large(5+)
  IsAlone          ← FamilySize      Binary: FamilySize == 1
  HasCabin         ← Cabin           Binary: Cabin is not null
  Deck             ← Cabin           First letter of Cabin (A-G)
  AgeMissing       ← Age             Binary: Age is null (before imputation)
  log(Fare+1)      ← Fare           

## 4b. Data Staging Strategy

Raw data is **immutable** — `data/raw/` is never modified. Each pipeline stage produces new output files in `data/processed/`, and reusable logic lives in `src/` modules that notebooks call.

```
data/
  raw/titanic/                  ← IMMUTABLE (selected in this task)
    train.csv                      891 rows × 12 cols (input to 3.2)
    test.csv                       418 rows × 11 cols (input to 3.2)
    gender_submission.csv          418 rows × 2 cols  (reference only)
  processed/                    ← Pipeline outputs
    train_clean.csv                Task 3.2: missing values imputed, types fixed
    test_clean.csv                 Task 3.2: same cleaning applied (fit on train)
    train_features.csv             Task 3.3: engineered features added
    test_features.csv              Task 3.3: same features applied

src/                            ← Reusable pipeline modules
  __init__.py
  cleaning.py                      clean_dataset(df) → cleaned DataFrame
  features.py                      build_features(df) → DataFrame with engineered features
```

**Pipeline flow:**
1. **3.1 Select Data** (this task) → selects `data/raw/titanic/{train,test}.csv`
2. **3.2 Clean Data** → reads raw, writes `data/processed/{train,test}_clean.csv` via `src/cleaning.py`
3. **3.3 Construct Data** → reads clean, writes `data/processed/{train,test}_features.csv` via `src/features.py`
4. **4.x Modeling** → reads `data/processed/{train,test}_features.csv`

**Guards:**
- All imputers/encoders/scalers fit on **train only**, then transform test
- `src/` functions are pure: DataFrame in → DataFrame out, no side effects
- Notebooks document decisions; `src/` modules implement them for reuse

## 5. Selection Dependencies

| Decision | Depends On | Notes |
|----------|-----------|-------|
| Age included despite 20% missing | Imputation in task 3.2 | Title-group median imputation (Mr~30, Mrs~36, Miss~22, Master~4) |
| Cabin included as engineering source | Deck/HasCabin extraction in task 3.3 | Raw Cabin too sparse (77% missing) for direct use |
| Embarked included despite 2 missing | Mode imputation in task 3.2 | Trivial fix (mode = 'S') |
| Fare included despite 1 missing in test | Pclass-median imputation in task 3.2 | Single missing value (PassengerId 1044) |
| All preprocessing | Fit on train only | Prevents preprocessing leakage into test set |

## Conclusions

**Data selection is complete.** All 3 datasets, all 12 fields, and all 1,309 records (891 train + 418 test) are retained.

Key takeaways:
1. **No fields excluded** — every field serves as target, ID, direct feature, or feature-engineering source
2. **No records excluded** — the dataset is too small (891 rows) to afford dropping observations; quality issues are handled via imputation in Phase 3.2
3. **No data leakage risks** — all features are pre-event attributes; train/test split is clean
4. **Critical guard:** all imputers/encoders must be fit on training data only

**Next step:** Run `/clean-data` (task 3.2) to handle missing values, outliers, and quality issues in the selected data.